In [ ]:
import os
import matplotlib.pyplot as plt 
import shutil
from ultralytics import YOLO
import torch
from PIL import Image
import random

In [190]:
# Splitting data
random.seed(42)

# Process each class
for class_name in classes:


    images_path = os.path.join(path, "Train", class_name, "images")
    labels_path = os.path.join(path, "Train", class_name, "labels")

    # Paths for the new test images and labels
    test_images_path = os.path.join(path, "Test", class_name, "images")
    test_labels_path = os.path.join(path, "Test", class_name, "labels")

    # Create test folders
    os.makedirs(test_images_path, exist_ok=True)
    os.makedirs(test_labels_path, exist_ok=True)

    # Get all training images
    images = os.listdir(images_path)

    # Randomly select 10% of the images for testing
    test_images = random.sample(images, int(len(images) * 0.10))

    # Move each selected image and its matching label to Test
    for image in test_images:
        label = os.path.splitext(image)[0] + ".txt"

        shutil.move(
            os.path.join(images_path, image),
            os.path.join(test_images_path, image)
        )

        shutil.move(
            os.path.join(labels_path, label),
            os.path.join(test_labels_path, label)
        )


In [191]:
import os

# Check the number of images and labels in each split
for split in ["Train", "Val", "Test"]:
    print(f"\n{split}:")

    for class_name in classes:
        images_path = os.path.join(path, split, class_name, "images")
        labels_path = os.path.join(path, split, class_name, "labels")

        images_count = len(os.listdir(images_path))
        labels_count = len(os.listdir(labels_path))

        print(f"  {class_name}: {images_count} images, {labels_count} labels")


Train:
  Glioma: 935 images, 935 labels
  Meningioma: 1175 images, 1175 labels
  No Tumor: 576 images, 576 labels
  Pituitary: 1151 images, 1151 labels

Val:
  Glioma: 136 images, 136 labels
  Meningioma: 140 images, 140 labels
  No Tumor: 97 images, 97 labels
  Pituitary: 136 images, 136 labels

Test:
  Glioma: 218 images, 218 labels
  Meningioma: 274 images, 274 labels
  No Tumor: 133 images, 133 labels
  Pituitary: 269 images, 269 labels


In [193]:
import os

# Check each dataset split and tumor class
for split in ["Train", "Val", "Test"]:
    print(f"\n{split}")

    for class_name in classes:
        images_path = os.path.join(path, split, class_name, "images")
        labels_path = os.path.join(path, split, class_name, "labels")

        images = [
            f for f in os.listdir(images_path)
            if os.path.isfile(os.path.join(images_path, f))
        ]

        labels = [
            f for f in os.listdir(labels_path)
            if f.endswith(".txt")
        ]

        print(
            f"{class_name}: "
            f"{len(images)} images, {len(labels)} labels"
        )


Train
Glioma: 935 images, 935 labels
Meningioma: 1175 images, 1175 labels
No Tumor: 576 images, 576 labels
Pituitary: 1151 images, 1151 labels

Val
Glioma: 136 images, 136 labels
Meningioma: 140 images, 140 labels
No Tumor: 97 images, 97 labels
Pituitary: 136 images, 136 labels

Test
Glioma: 218 images, 218 labels
Meningioma: 274 images, 274 labels
No Tumor: 133 images, 133 labels
Pituitary: 269 images, 269 labels


In [ ]:
# creating a folder structure compatible with yolo ultralytics 



# Original dataset
source_path = "/workspace/dataset"

# New YOLO-compatible dataset
output_path = "/workspace/dataset_yolo"

classes = ["Glioma", "Meningioma", "No Tumor", "Pituitary"]
splits = ["Train", "Val", "Test"]

# Remove the previous output folder if it exists
if os.path.exists(output_path):
    shutil.rmtree(output_path)

# Create YOLO folders
for split in ["train", "val", "test"]:
    os.makedirs(os.path.join(output_path, "images", split))
    os.makedirs(os.path.join(output_path, "labels", split))

# Copy images and matching labels
for split in splits:
    split_name = split.lower()

    for class_name in classes:
        images_path = os.path.join(
            source_path, split, class_name, "images"
        )
        labels_path = os.path.join(
            source_path, split, class_name, "labels"
        )

        for image in os.listdir(images_path):

            # Get the matching label filename
            label = os.path.splitext(image)[0] + ".txt"

            # Add the class name to avoid filename collisions
            new_name = f"{class_name.replace(' ', '_')}_{image}"
            new_label = os.path.splitext(new_name)[0] + ".txt"

            # Copy the image
            shutil.copy2(
                os.path.join(images_path, image),
                os.path.join(output_path, "images", split_name, new_name)
            )

            # Copy its matching label
            shutil.copy2(
                os.path.join(labels_path, label),
                os.path.join(output_path, "labels", split_name, new_label)
            )


print("Location:", output_path)


YOLO dataset created successfully!
Location: /workspace/dataset_yolo


In [199]:

from ultralytics import YOLO
import os
import time

# Load the pretrained YOLOv8 nano model
model = YOLO("yolov8n.pt")

# Path to your data.yaml in the project folder
data_yaml = "/workspace/NeuroVision_AI/notebooks/data.yaml"

# Start a 1-epoch test
start_time = time.time()

results = model.train(
    data=data_yaml,
    epochs=1,
    imgsz=640,
    batch=8,
    patience=10,
    project="runs/detect",
    name="brain_tumor_test",
    exist_ok=True
)

# Calculate training duration
duration = time.time() - start_time

print(f"Test training completed in {duration:.2f} seconds")


Ultralytics 8.4.191 🚀 Python-3.10.12 torch-2.14.1+cu130 CUDA:0 (NVIDIA RTX A4000, 15974MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/workspace/NeuroVision_AI/notebooks/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=brain_tumor_test, nbs=64, nms=None, ops

: 

: 

: 